In [ ]:
import pandas as pd

# 读取 data 文件夹中的 Excel 表格
df = pd.read_excel("data/data.xlsx", sheet_name="data")

# 查看数据规模和所有列名
print("数据规模（行数、列数）：", df.shape)
print("所有列名：", df.columns.tolist())

# 显示前5行数据
display(df.head())

In [ ]:
# 一、查看数据结构、缺失值和重复记录
df.info()

# 统计每一列的缺失数量和比例
missing = pd.DataFrame({
    "缺失数量": df.isna().sum(),
    "缺失比例（%）": (df.isna().mean() * 100).round(2)
})
display(missing.sort_values("缺失数量", ascending=False))

print("完全重复的行数：", df.duplicated().sum())
print("商品ID重复的行数：", df["商品ID"].dropna().duplicated().sum())

In [ ]:
# 二、查看价格、类目和销量的原始写法
price = pd.to_numeric(df["价格（元）"], errors="coerce")

print("价格统计")
display(price.describe())

print("各类目的记录数量")
display(df["分类"].value_counts(dropna=False))

print("销量原始写法及出现次数（前15种）")
display(df["销量（件）"].value_counts(dropna=False).head(15))

In [ ]:
# 三、绘制价格分布和类目数量图
import matplotlib.pyplot as plt
import seaborn as sns

# 设置中文字体
plt.rcParams["font.sans-serif"] = ["SimHei"]
plt.rcParams["axes.unicode_minus"] = False

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 使用非空的数值价格绘图
sns.histplot(price.dropna(), bins=50, ax=axes[0])
axes[0].set_title("商品价格分布")
axes[0].set_xlabel("价格（元）")
axes[0].set_ylabel("记录数量")

# 展示原始表中记录数量最多的10个类目
category_counts = df["分类"].value_counts().head(10)
category_counts.sort_values().plot(kind="barh", ax=axes[1], color="steelblue")
axes[1].set_title("记录数量最多的10个类目")
axes[1].set_xlabel("记录数量")
axes[1].set_ylabel("商品类目")

plt.tight_layout()
plt.show()

In [ ]:
# 四、按类目核对价格和销量的完整性
availability = df.groupby("分类", dropna=False).agg(
    总记录数=("ID", "size"),
    价格非空数=("价格（元）", "count"),
    销量非空数=("销量（件）", "count")
)
display(availability.sort_values("销量非空数", ascending=False))

# 统计同时具有数值价格和销量原文的记录
complete_mask = price.notna() & df["销量（件）"].notna()
print("同时具有数值价格和销量原文的记录数：", complete_mask.sum())

In [ ]:
# 五、清洗商品付款人数数据
import pandas as pd

# 读取原始文件，商品ID按字符串保存
raw = pd.read_excel(
    "data/data.xlsx",
    sheet_name="data",
    dtype={"商品ID": "string"}
)
cleaned = raw.copy()

# 将价格转换为数字
cleaned["price"] = pd.to_numeric(cleaned["价格（元）"], errors="coerce")

# 提取付款人数文本中的数字、万字单位和加号
text = cleaned["销量（件）"].astype("string").str.strip()
parts = text.str.extract(r"^(\d+(?:\.\d+)?)(万)?(\+)?人付款$")
number = pd.to_numeric(parts[0], errors="coerce")
multiplier = parts[1].map({"万": 10000}).fillna(1)

# 展示值保留原始计量含义，加号单独标记
cleaned["payment_display"] = number * multiplier
cleaned["payment_has_plus"] = parts[2].notna()

# 筛选正价格且付款人数格式可识别的记录
cleaned = cleaned.loc[
    cleaned["price"].gt(0) & cleaned["payment_display"].notna()
].copy()

# 重复商品ID先标记，后续划分数据时按商品分组
cleaned["product_id_repeated"] = cleaned["商品ID"].duplicated(keep=False)

print("原始记录数：", len(raw))
print("清洗后记录数：", len(cleaned))
print("排除浏览人数记录数：", text.str.endswith("人看过", na=False).sum())
print("保留展示付款人数为0的记录数：", cleaned["payment_display"].eq(0).sum())
print("保留带加号标记的记录数：", cleaned["payment_has_plus"].sum())
display(cleaned[[
    "商品ID", "分类", "price", "销量（件）", "payment_display",
    "payment_has_plus", "product_id_repeated"
]].head(10))


In [ ]:
# 六、保存清洗结果并重新读取核验
cleaned.to_csv("data/sales_cleaned.csv", index=False, encoding="utf-8-sig")

# 重新读取，核对记录数、字段、商品ID与付款人数展示值
saved = pd.read_csv("data/sales_cleaned.csv", dtype={"商品ID": "string"})
assert len(saved) == len(cleaned)
assert saved.columns.tolist() == cleaned.columns.tolist()
pd.testing.assert_series_equal(
    saved["商品ID"], cleaned["商品ID"].reset_index(drop=True), check_dtype=False
)
pd.testing.assert_series_equal(
    saved["payment_display"], cleaned["payment_display"].reset_index(drop=True),
    check_dtype=False
)
assert saved["销量（件）"].str.endswith("人付款", na=False).all()
assert saved["price"].gt(0).all()
assert saved["payment_display"].ge(0).all()
assert saved["payment_has_plus"].tolist() == cleaned["payment_has_plus"].tolist()
assert saved["product_id_repeated"].tolist() == cleaned["product_id_repeated"].tolist()

print("已保存到 data/sales_cleaned.csv")
print(f"CSV核验通过，共{len(saved)}条记录、{len(saved.columns)}列。")
print("字段含义：payment_display为付款人数展示值，统计周期仍待数据源确认。")


In [ ]:
# 七、确定练习目标与输入特征
import pandas as pd

# 独立读取清洗结果，商品ID用于后续按商品分组
model_data = pd.read_csv("data/sales_cleaned.csv", dtype={"商品ID": "string"})
target_column = "payment_display"
target_description = "付款人数展示值（统计周期未知，带加号的数值为近似展示值）"

# 当前486条样本均为电脑商品，选用5个具有变化的输入字段
numeric_features = ["price", "是否包邮"]
categorical_features = ["发货省份", "发货城市", "价格描述"]
feature_columns = numeric_features + categorical_features
X = model_data[feature_columns].copy()

# 包邮转换为0/1，类别缺失值使用固定的未知类别
X["price"] = pd.to_numeric(X["price"], errors="coerce")
X["是否包邮"] = X["是否包邮"].map({"包邮": 1, "不包邮": 0})
for column in categorical_features:
    X[column] = X[column].fillna("未知").astype(str)

# 销量原文和加号标记用于解释目标，商品ID用于后续分组
# 输入字段由feature_columns明确指定，避免目标信息进入输入
# 统计周期仍需数据集说明确认，当前先进行展示值回归练习
y = pd.to_numeric(model_data[target_column], errors="raise").rename(target_column)
groups = model_data["商品ID"]
assert y.notna().all() and y.ge(0).all(), "目标值需要完整且非负。"
assert groups.notna().all(), "按商品分组需要完整的商品ID。"

feature_summary = pd.DataFrame({
    "输入字段": feature_columns,
    "数据类型": ["数值", "0/1", "类别", "类别", "类别"],
    "处理规则": ["训练集拟合中位数填充", "包邮=1，不包邮=0", "独热编码", "缺失填未知后独热编码", "独热编码"]
})
print("当前练习目标为" + target_description)
print(f"输入特征矩阵为{len(X)}行、{X.shape[1]}列，目标值共{len(y)}个。")
print(f"后续按商品ID划分数据，共{groups.nunique()}个商品组。")
display(feature_summary)
display(X.head())

In [ ]:
# 八、配置特征编码器，并演示前5行编码效果
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# 数值填充与类别字典由后续训练集学习
# handle_unknown允许测试集和新商品出现新的省份、城市或价格描述
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", SimpleImputer(strategy="median"), numeric_features),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features)
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

# 使用独立副本演示前5行，正式编码器保留给下一步的训练集
preview_encoder = clone(preprocessor).set_output(transform="pandas")
encoded_preview = preview_encoder.fit_transform(X.head(5))
print("特征准备完成，5个输入字段的处理规则和编码器已配置。")
print(f"前5行编码演示得到{encoded_preview.shape[1]}个数值列。")
display(encoded_preview)
print("下一步在训练集上调用preprocessor.fit_transform，测试集使用preprocessor.transform。")
print("当前练习目标为付款人数展示值，月销量统计周期仍待来源说明确认。")

In [ ]:
# 九、按商品ID划分训练集与测试集
from sklearn.model_selection import GroupShuffleSplit

# 一件商品可能有多条记录，必须整组分配，避免同一商品跨两组
# test_size=0.2表示抽取约20%的商品组，记录比例不一定恰好为20%
# 固定random_state后，每次运行可得到同样的划分结果
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_indices, test_indices = next(splitter.split(X, y, groups=groups))

# 同一套位置索引同时切分输入、目标值和商品ID，确保逐行对应
X_train = X.iloc[train_indices].copy()
X_test = X.iloc[test_indices].copy()
y_train = y.iloc[train_indices].copy()
y_test = y.iloc[test_indices].copy()
groups_train = groups.iloc[train_indices].copy()
groups_test = groups.iloc[test_indices].copy()

# 检查两组无交叉、无遗漏，且同一商品不会同时进入两组
train_product_ids = set(groups_train)
test_product_ids = set(groups_test)
overlap_product_ids = train_product_ids & test_product_ids
assert not overlap_product_ids, "训练集与测试集存在重复商品。"
assert set(train_indices).isdisjoint(test_indices), "两组记录位置出现交叉。"
assert set(train_indices) | set(test_indices) == set(range(len(X))), "划分遗漏了记录。"
assert X_train.index.equals(y_train.index) and X_test.index.equals(y_test.index)

split_summary = pd.DataFrame({
    "数据集": ["训练集", "测试集"],
    "记录数": [len(X_train), len(X_test)],
    "不同商品数": [groups_train.nunique(), groups_test.nunique()],
    "记录占比（%）": [round(len(X_train) / len(X) * 100, 2),
                       round(len(X_test) / len(X) * 100, 2)]
})
print("按商品ID分组划分完成，随机种子为42。")
display(split_summary)
print(f"两组重复商品数：{len(overlap_product_ids)}，检查通过。")
print(f"全部{len(X)}条记录均已分配，输入与目标值逐行对应。")
print("训练集用于学习；测试集留给后续最终评估。")

In [ ]:
# 十、仅用训练集学习填充规则和独热编码
import numpy as np
from sklearn.base import clone

# 重新创建正式编码器；前面5行演示用的preview_encoder不参与正式处理
preprocessor = clone(preprocessor).set_output(transform="pandas")

# fit_transform：在训练集学习中位数、类别列表，并转换训练集
X_train_encoded = preprocessor.fit_transform(X_train)

# transform：测试集只使用已学好的规则，不能再次fit
# 新类别由handle_unknown="ignore"处理，不会改变输出列的顺序
X_test_encoded = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out().tolist()

# 检查编码结果完整、有限，且两组使用完全相同的输入列
assert X_train_encoded.columns.tolist() == X_test_encoded.columns.tolist()
assert X_train_encoded.columns.tolist() == feature_names
assert X_train_encoded.index.equals(y_train.index)
assert X_test_encoded.index.equals(y_test.index)
assert np.isfinite(X_train_encoded.to_numpy()).all(), "训练特征含缺失或无穷值。"
assert np.isfinite(X_test_encoded.to_numpy()).all(), "测试特征含缺失或无穷值。"
assert preprocessor.transformers_[0][1].statistics_[0] == X_train["price"].median()

# 在页面展示最终状态，便于下一步直接训练三个回归模型
training_ready_summary = split_summary.copy()
training_ready_summary["编码后特征数"] = [X_train_encoded.shape[1], X_test_encoded.shape[1]]
print("第1项完成：按商品ID划分，且正式编码器仅在训练集上拟合。")
display(training_ready_summary)
print(f"重复商品数：{len(overlap_product_ids)}；特征无缺失、无无穷值，检查通过。")
print("训练输入：X_train_encoded；训练目标：y_train。")
print("测试输入：X_test_encoded；测试目标：y_test。")
print("训练集编码预览（前5行、前8列）：")
display(X_train_encoded.iloc[:5, :8])
print("下一步：训练线性回归、随机森林和XGBoost。")
print("当前目标为付款人数展示值，统计周期未知。")

In [ ]:
# 十一、导入并创建三个回归模型
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# 先确认前面的数据准备代码已经运行
print("训练特征规模：", X_train_encoded.shape)
print("训练目标数量：", len(y_train))

# 字典用于统一管理三个模型
models = {
    # 模型1：线性回归，作为基础对照模型
    "线性回归": LinearRegression(),

    # 模型2：随机森林，多棵决策树共同预测
    "随机森林": RandomForestRegressor(
        n_estimators=200,     # 建立200棵树
        max_depth=8,          # 限制每棵树的最大深度
        min_samples_leaf=2,  # 每个叶节点至少保留2条记录
        random_state=42,     # 固定随机种子，方便重复实验
        n_jobs=2             # 使用2个CPU线程
    ),

    # 模型3：XGBoost，逐轮加入树来改善预测
    "XGBoost": XGBRegressor(
        objective="reg:squarederror",  # 使用平方误差回归目标
        n_estimators=200,              # 进行200轮提升
        max_depth=3,                   # 控制树的深度
        learning_rate=0.05,            # 每轮更新的步长
        subsample=0.8,                 # 每轮随机使用80%的训练记录
        colsample_bytree=0.8,          # 每棵树随机使用80%的特征
        tree_method="hist",           # 使用直方图建树方法
        device="cpu",                 # 使用CPU
        random_state=42,
        n_jobs=2
    )
}

print("三个模型已经创建，接下来开始训练。")

In [ ]:
# 十二、使用同一份训练数据，依次训练三个模型
from time import perf_counter
from sklearn.utils.validation import check_is_fitted

training_records = []

for name, model in models.items():
    print(f"开始训练：{name}", flush=True)
    start_time = perf_counter()

    # fit表示拟合：模型从训练特征和对应目标值中学习规律
    model.fit(X_train_encoded, y_train)

    elapsed = perf_counter() - start_time

    # 确认模型已经拟合，并且使用了全部训练特征
    check_is_fitted(model)
    assert model.n_features_in_ == X_train_encoded.shape[1]

    training_records.append({
        "模型": name,
        "训练记录数": len(y_train),
        "输入特征数": model.n_features_in_,
        "耗时（秒）": round(elapsed, 3),
        "状态": "训练完成"
    })

    print(f"{name}训练完成，耗时{elapsed:.3f}秒。\n")

# 汇总三个模型的训练状态；耗时不代表预测效果好坏
training_summary = pd.DataFrame(training_records)
display(training_summary)
print("三个模型均已训练完成，保存在当前内核的models字典中。")
print("本步只使用训练集学习，测试集留给下一步评估MAE、RMSE。")
print("当前目标为付款人数展示值，统计周期未知。")

In [ ]:
# 十三、在保留测试集上计算MAE与RMSE
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.utils.validation import check_is_fitted

# 沿用此前固定的商品分组划分，不在测试集上拟合或调整模型
assert set(groups_train).isdisjoint(set(groups_test))
assert X_test_encoded.index.equals(y_test.index)
assert X_test_encoded.columns.tolist() == X_train_encoded.columns.tolist()

evaluation_records = []
predictions = {}
test_predictions = pd.DataFrame({
    "商品ID": groups_test,
    "真实展示值": y_test
})

for name, model in models.items():
    check_is_fitted(model)
    y_pred = np.asarray(model.predict(X_test_encoded), dtype=float).reshape(-1)
    assert len(y_pred) == len(y_test) and np.isfinite(y_pred).all()

    # 使用原始预测计算指标，不通过截断负数来改变当前评估口径
    predictions[name] = y_pred
    test_predictions[name + "预测值"] = y_pred
    evaluation_records.append({
        "模型": name,
        "测试记录数": len(y_test),
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "负预测数": int((y_pred < 0).sum())
    })

results_df = pd.DataFrame(evaluation_records)
print(f"测试集评估完成：{len(y_test)}条记录、{groups_test.nunique()}个不同商品。")
display(results_df.round({"MAE": 2, "RMSE": 2}))
print("MAE为平均绝对误差；RMSE为均方根误差，两项均越小越好。")
print("负预测数单独报告，全部指标保留原始预测口径。")
print("目标仍为付款人数展示值，统计周期未知，不能解释为精确月销量误差。")

In [ ]:
# 十四、加入简单参考，保存评估表和逐条预测
from pathlib import Path

# 参考值只由训练集计算；不使用测试目标来调整模型
baseline_records = []
for name, constant in {
    "训练集均值参考": float(y_train.mean()),
    "训练集中位数参考": float(y_train.median())
}.items():
    baseline_pred = np.full(len(y_test), constant)
    baseline_records.append({
        "模型": name,
        "预测常数": constant,
        "MAE": mean_absolute_error(y_test, baseline_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, baseline_pred))
    })
baseline_results_df = pd.DataFrame(baseline_records)

# 放在同一张表，展示模型与简单参考的误差
evaluation_summary = pd.concat([
    results_df[["模型", "MAE", "RMSE"]],
    baseline_results_df[["模型", "MAE", "RMSE"]]
], ignore_index=True)
print("95条测试记录的评估汇总（误差越小越好）：")
display(evaluation_summary.round({"MAE": 2, "RMSE": 2}))

lowest_mae_model = results_df.loc[results_df["MAE"].idxmin(), "模型"]
lowest_rmse_model = results_df.loc[results_df["RMSE"].idxmin(), "模型"]
print(f"三个回归模型中，本次MAE最低：{lowest_mae_model}；RMSE最低：{lowest_rmse_model}。")
print("这是固定测试集上的观察结果，后续调参需在训练集内部进行分组交叉验证。")
mean_reference = baseline_results_df.loc[baseline_results_df["模型"].eq("训练集均值参考")].iloc[0]
median_reference = baseline_results_df.loc[baseline_results_df["模型"].eq("训练集中位数参考")].iloc[0]
if results_df["RMSE"].min() >= mean_reference["RMSE"]:
    print("三个模型的RMSE均高于或等于训练集均值参考，当前未见相对该参考的改善。")
if results_df["MAE"].min() >= median_reference["MAE"]:
    print("三个模型的MAE均高于或等于训练集中位数参考。")
print("负预测数：" + "；".join(f"{row['模型']} {row['负预测数']}条" for _, row in results_df.iterrows()))

# 保留完整精度，导出后可用于后续画图或填写README
results_dir = Path("results")
results_dir.mkdir(exist_ok=True)
results_df.to_csv(results_dir / "model_metrics.csv", index=False, encoding="utf-8-sig")
baseline_results_df.to_csv(results_dir / "baseline_metrics.csv", index=False, encoding="utf-8-sig")
test_predictions.to_csv(results_dir / "test_predictions.csv", index=True,
                        index_label="原记录索引", encoding="utf-8-sig")
print("已保存results/model_metrics.csv、baseline_metrics.csv、test_predictions.csv。")
print("目标为付款人数展示值，统计周期未知；带加号的原始展示数值为近似下界。")

In [ ]:
# 十五、绘制MAE和RMSE对比图，保留两个简单参考
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager

# 优先使用本机可用的中文字体，避免中文显示为方框
installed_fonts = {font.name for font in font_manager.fontManager.ttflist}
chinese_font = next((name for name in ["Microsoft YaHei", "SimHei", "Noto Sans CJK SC"]
                     if name in installed_fonts), "DejaVu Sans")
plt.rcParams.update({"font.sans-serif": [chinese_font], "axes.unicode_minus": False,
                     "font.size": 11, "figure.dpi": 110, "savefig.dpi": 300})
img_dir = Path("img")
img_dir.mkdir(exist_ok=True)

# 直接使用已保存的评估值，不重新训练或调整模型
comparison_plot = pd.concat([
    pd.read_csv("results/model_metrics.csv")[["模型", "MAE", "RMSE"]],
    pd.read_csv("results/baseline_metrics.csv")[["模型", "MAE", "RMSE"]]
], ignore_index=True)
assert np.isfinite(comparison_plot[["MAE", "RMSE"]].to_numpy()).all()
colors = ["#2563eb", "#0f766e", "#d97706", "#94a3b8", "#cbd5e1"]
positions = np.arange(len(comparison_plot))
fig, axes = plt.subplots(1, 2, figsize=(11.8, 6.5), sharey=True)

for ax, metric in zip(axes, ["MAE", "RMSE"]):
    values = comparison_plot[metric].to_numpy()
    bars = ax.barh(positions, values, color=colors, height=0.58)
    for bar, name in zip(bars, comparison_plot["模型"]):
        if "参考" in name:
            bar.set_hatch("//")
            bar.set_edgecolor("#64748b")
    ax.bar_label(bars, fmt="%.2f", padding=5, fontsize=10, color="#0f172a")
    ax.set_title(metric + "（越小越好）", fontsize=13, fontweight="bold", pad=14)
    ax.set_xlabel("误差（付款人数展示值）", labelpad=10)
    ax.set_xlim(0, values.max() * 1.28)
    ax.set_axisbelow(True)
    ax.grid(axis="x", color="#e2e8f0", linewidth=0.8)
    ax.tick_params(axis="y", length=0, pad=10)
    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_color("#cbd5e1")

axes[0].set_yticks(positions, comparison_plot["模型"])
axes[0].invert_yaxis()
fig.suptitle("商品付款人数展示值：测试集误差对比", fontsize=17, fontweight="bold", y=0.97)
fig.text(0.5, 0.90, "95条测试记录 / 93个商品组 · 按商品ID分组划分",
         ha="center", fontsize=11, color="#475569")
fig.text(0.03, 0.09, "灰色斜线为简单参考：每次都预测训练集均值或中位数。", fontsize=10, color="#475569")
fig.text(0.03, 0.045, "当前三个模型均未优于简单参考；目标统计周期未知，不能解释为精确月销量误差。",
         fontsize=10, color="#475569")
fig.subplots_adjust(left=0.20, right=0.97, top=0.80, bottom=0.23, wspace=0.18)
comparison_image_path = img_dir / "model_comparison.png"
fig.savefig(comparison_image_path, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
plt.close(fig)
print("模型对比图已保存：" + str(comparison_image_path))

In [ ]:
# 十六、绘制XGBoost特征重要性，并按原字段汇总
from matplotlib.ticker import PercentFormatter

# gain表示某特征参与分裂时带来的平均训练损失下降
# 使用已经训练好的模型，不重新拟合，也不依赖测试目标
xgb_booster = models["XGBoost"].get_booster()
assert xgb_booster.feature_names == feature_names, "模型特征顺序与编码列不一致。"
raw_gain = pd.Series(xgb_booster.get_score(importance_type="gain"), dtype=float)
raw_gain = raw_gain.reindex(feature_names, fill_value=0.0)
assert np.isfinite(raw_gain.to_numpy()).all() and raw_gain.sum() > 0
normalized_gain = raw_gain / raw_gain.sum()

# 将独热编码后的各列对应回5个原始输入字段
original_feature_names = numeric_features + categorical_features
feature_sources = []
for encoded_name in feature_names:
    matches = [name for name in original_feature_names
               if encoded_name == name or encoded_name.startswith(name + "_")]
    assert len(matches) == 1, "无法唯一确定编码列对应的原字段。"
    feature_sources.append(matches[0])

feature_importance_df = pd.DataFrame({
    "编码特征": feature_names, "原始字段": feature_sources,
    "归一化平均增益": normalized_gain.to_numpy()
}).sort_values("归一化平均增益", ascending=False, ignore_index=True)
grouped_importance_df = (feature_importance_df.groupby("原始字段", as_index=False)
                         ["归一化平均增益"].sum()
                         .sort_values("归一化平均增益", ascending=False, ignore_index=True))
assert np.isclose(feature_importance_df["归一化平均增益"].sum(), 1.0)
assert np.isclose(grouped_importance_df["归一化平均增益"].sum(), 1.0)

# 左边展示编码列Top12，右边将同一原字段的编码列重要性相加
top_encoded = feature_importance_df.head(12).sort_values("归一化平均增益")
grouped_plot = grouped_importance_df.sort_values("归一化平均增益")
field_colors = dict(zip(original_feature_names,
                       ["#2563eb", "#7c3aed", "#0f766e", "#0891b2", "#d97706"]))
fig, axes = plt.subplots(1, 2, figsize=(12.6, 7.6))
for ax, table, label_column, title in [
    (axes[0], top_encoded, "编码特征", "编码特征 Top 12"),
    (axes[1], grouped_plot, "原始字段", "原字段汇总（编码列求和）")
]:
    values = table["归一化平均增益"].to_numpy()
    bars = ax.barh(table[label_column], values, height=0.65,
                   color=[field_colors[name] for name in table["原始字段"]])
    ax.bar_label(bars, labels=[f"{value:.1%}" for value in values], padding=5, fontsize=10)
    ax.xaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=0))
    ax.set_xlim(0, max(values.max() * 1.35, 0.05))
    ax.set_title(title, fontsize=13, fontweight="bold", pad=14)
    ax.set_xlabel("归一化平均增益", labelpad=10)
    ax.set_axisbelow(True)
    ax.grid(axis="x", color="#e2e8f0", linewidth=0.8)
    ax.tick_params(axis="y", length=0, pad=8, labelsize=10)
    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_color("#cbd5e1")

fig.suptitle("XGBoost 特征重要性", fontsize=18, fontweight="bold", y=0.97)
fig.text(0.5, 0.91, "已训练模型 / gain平均增益 / 全部56个编码特征归一化后合计100%",
         ha="center", fontsize=11, color="#475569")
fig.text(0.025, 0.10, "右图为编码列重要性之和，仅作模型内部参考；左图只展示Top12，百分比不要求合计100%。",
         fontsize=10, color="#475569")
fig.text(0.025, 0.055, "重要性不代表因果关系或影响方向；当前模型在测试集上未优于简单参考。",
         fontsize=10, color="#475569")
fig.subplots_adjust(left=0.17, right=0.97, top=0.81, bottom=0.23, wspace=0.67)
importance_image_path = img_dir / "feature_importance.png"
fig.savefig(importance_image_path, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
plt.close(fig)

# 保存完整56列及5个原始字段的重要性数值，便于核对与写报告
feature_importance_df.to_csv("results/xgboost_feature_importance.csv", index=False, encoding="utf-8-sig")
grouped_importance_df.to_csv("results/xgboost_grouped_importance.csv", index=False, encoding="utf-8-sig")
print("特征重要性图已保存：" + str(importance_image_path))
print("完整特征重要性与字段汇总表已保存到results目录。")
display(grouped_importance_df.round({"归一化平均增益": 4}))